# U08 查詢處理與最佳化：打造迷你 SQL 引擎

**資料庫管理**・11/05　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

一句 SQL 進去之後到底發生什麼事？今天**從零手造**：tokenizer → parser → AST → volcano 執行器；然後看 join 三演算法對決、最佳化器怎麼「猜」、以及 DuckDB 為什麼在分析題快十倍。

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 | 帶走什麼 |
|---|---|---|---|
| 第 1 節 | 50 | SQL 的一生・關聯代數・**mini SQL 引擎一路做到 `GROUP BY`**・volcano 模型 | 「宣告式」的魔術拆穿了 |
| 第 2 節 | 50 | join 三演算法實測・join 順序成本試算・估計為何會錯（偏斜／相關）・DuckDB 列式對決 | 看懂計畫、救得了慢查詢 |
| 課堂實作 | 35 | 計畫閱讀＋join 量測＋把 EXPLAIN 用回你的專題 | 專題效能收尾 |

> 時間配置仍是 **50＋50＋35＝135 分鐘**；標成「選讀」的延伸不計入課堂時間。
> 期中考週剛結束——本單元與下一單元是「懂資料庫」支線的高潮，**不要求你把引擎做成完整產品**，但親手跑過核心路徑後，你會更有把握解釋專題裡的計畫。

# 第 1 節：SQL 的一生

## 1.1 從字串到答案的流水線

```
 "SELECT name FROM student WHERE year >= 3"
        │ tokenizer（切詞）
        ▼
 [KW SELECT] [ID name] [KW FROM] [ID student] [KW WHERE] [ID year] [OP >=] [NUM 3]
        │ parser（遞迴下降 → 語法樹 AST）
        ▼
 {select:[name], from:student, where:[(year, >=, 3)]}
        │ 翻成關聯代數 → 最佳化器改寫（挑順序、挑索引）
        ▼
 π_name( σ_{year≥3}( student ) )        ← 邏輯計畫
        │ 選實體演算法（掃描？走索引？哪種 join？）
        ▼
 volcano 執行器：一列一列往上「拉」        ← 實體計畫
```

U01 說過「SQL 說 what，系統決定 how」——今天就把「系統」造出來，魔術就不再是魔術。

## 1.2 夠用的關聯代數：四個運算子撐起整個 SQL

| 代數 | 讀法 | SQL 對應 |
|---|---|---|
| σ（selection） | 挑**列** | `WHERE` |
| π（projection） | 挑**欄** | `SELECT a, b` |
| ⋈（join） | 接表 | `JOIN … ON` |
| γ（grouping） | 分堆聚合 | `GROUP BY` |

它們**吃表、吐表**——所以可以任意套疊（closure property）。先用 20 行 Python 把四個都做出來：

In [ ]:
# 關聯代數四件套：表 = list of dict
students = [
    {"sid": "S001", "name": "林佳蓉", "dept": "統計", "year": 3},
    {"sid": "S002", "name": "陳威廷", "dept": "統計", "year": 3},
    {"sid": "S004", "name": "李承翰", "dept": "統計", "year": 2},
    {"sid": "S007", "name": "吳孟軒", "dept": "資訊", "year": 3},
    {"sid": "S010", "name": "許芷瑄", "dept": "數學", "year": 3},
    {"sid": "S012", "name": "謝欣妤", "dept": "數學", "year": 4},
]
takes = [
    {"sid": "S001", "cid": "C101", "grade": 88}, {"sid": "S001", "cid": "C104", "grade": 92},
    {"sid": "S002", "cid": "C101", "grade": 76}, {"sid": "S007", "cid": "C301", "grade": 93},
    {"sid": "S010", "cid": "C201", "grade": 90}, {"sid": "S012", "cid": "C201", "grade": 71},
]

def sel(rows, pred):                                    # σ selection：挑列
    return [r for r in rows if pred(r)]

def proj(rows, cols):                                   # π projection：挑欄
    return [{c: r[c] for c in cols} for r in rows]

def join_h(A, B, key):                                  # ⋈ join（hash 版，第 2 節拆效能）
    ht = {}
    for b in B:
        ht.setdefault(b[key], []).append(b)
    return [{**a, **b} for a in A for b in ht.get(a[key], [])]

def grp(rows, key, agg_fn):                             # γ grouping：分堆 → 每堆一列
    groups = {}
    for r in rows:
        groups.setdefault(r[key], []).append(r)
    return [{key: k, **agg_fn(rs)} for k, rs in groups.items()]

# 手排一句「SQL」：統計系學生的各人平均成績
result = grp(join_h(sel(students, lambda r: r["dept"] == "統計"), takes, "sid"),
             "sid", lambda rs: {"avg": sum(r["grade"] for r in rs) / len(rs)})
for r in result:
    print(r)

In [ ]:
# 代數版 vs SQLite 版——答案必須一模一樣（我們在驗證自己的引擎！）
import sqlite3
vcon = sqlite3.connect(":memory:")
vcon.execute("CREATE TABLE student(sid TEXT, name TEXT, dept TEXT, year INTEGER)")
vcon.execute("CREATE TABLE takes(sid TEXT, cid TEXT, grade INTEGER)")
vcon.executemany("INSERT INTO student VALUES (:sid, :name, :dept, :year)", students)
vcon.executemany("INSERT INTO takes VALUES (:sid, :cid, :grade)", takes)

sql_ans = dict(vcon.execute("""
    SELECT t.sid, AVG(t.grade) FROM student s JOIN takes t ON s.sid = t.sid
    WHERE s.dept = '統計' GROUP BY t.sid""").fetchall())
alg_ans = {r["sid"]: r["avg"] for r in result}
print("SQLite：", sql_ans)
print("代數版：", alg_ans)
assert sql_ans == alg_ans
print("✅ 一致——SQL 只是這四個運算子的「好看寫法」；最佳化器改寫的就是這棵代數樹（2.3 見）")

## 1.3 現場實作：mini SQL 引擎

第一版規格（單表版；join 你剛在代數層做過了，1.4 再加入分組聚合）：

> `SELECT 欄位們|* FROM 表 [WHERE 條件 AND 條件…] [ORDER BY 欄 [DESC]] [LIMIT n]`
> 條件 ＝ `欄位 運算子 常數`；運算子 ＝ `= <> != < > <= >=`

三個零件先用短版看懂骨架：**tokenizer**（字串→詞）、**parser**（詞→語法樹）、**executor**（語法樹→答案）；1.4 再沿著同一骨架補上分組聚合。

In [ ]:
# 零件 1：tokenizer——正規表達式把 SQL 字串切成「詞」
import re

TOKEN_RULES = [("NUM", r"\d+(?:\.\d+)?"),           # 數字
        ("STR", r"'[^']*'"),                      # '字串'
        ("ID",  r"[A-Za-z_\u4e00-\u9fff][A-Za-z0-9_\u4e00-\u9fff]*"),   # 識別字（也容忍中文）
        ("OP",  r"<=|>=|<>|!=|=|<|>"),            # 比較運算子（長的要放前面！）
        ("PUNC", r"[,*()]"),
        ("WS",  r"\s+")]                          # 空白（切完就丟）
MASTER_RE = re.compile("|".join(f"(?P<{n}>{p})" for n, p in TOKEN_RULES))
KEYWORDS = {"SELECT", "FROM", "WHERE", "AND", "GROUP", "ORDER", "BY",
            "LIMIT", "DESC", "ASC"}

def tokenize(sql):
    out, pos = [], 0
    for m in MASTER_RE.finditer(sql):
        if m.start() != pos:
            raise SyntaxError(f"看不懂的字元：{sql[pos:m.start()]!r}")
        pos = m.end()
        kind, val = m.lastgroup, m.group()
        if kind == "WS":
            continue
        if kind == "ID" and val.upper() in KEYWORDS:
            out.append(("KW", val.upper()))
        elif kind == "NUM":
            out.append(("NUM", float(val) if "." in val else int(val)))
        elif kind == "STR":
            out.append(("STR", val[1:-1]))
        else:
            out.append((kind, val))
    if pos != len(sql):
        raise SyntaxError(f"看不懂的結尾：{sql[pos:]!r}")
    return out

for tk in tokenize("SELECT name, year FROM student WHERE dept = '統計' AND year >= 3"):
    print(tk, end="  ")
print()

In [ ]:
# 零件 2：parser——遞迴下降：每條文法規則一個函數，讀到什麼就「吃」什麼
class Parser:
    def __init__(self, tokens):
        self.t, self.i = tokens, 0

    def peek(self):
        return self.t[self.i] if self.i < len(self.t) else ("EOF", None)

    def eat(self, kind, val=None):
        k, v = self.peek()
        if k != kind or (val is not None and v != val):
            raise SyntaxError(f"預期 {kind} {val or ''}，看到 {k} {v}")
        self.i += 1
        return v

    def parse(self):                                    # 文法的最上層
        self.eat("KW", "SELECT")
        items = self.select_list()
        self.eat("KW", "FROM")
        tbl = self.eat("ID")
        ast = {"select": items, "from": tbl, "where": [], "group": None,
               "order": None, "limit": None}
        if self.peek() == ("KW", "WHERE"):
            self.eat("KW", "WHERE")
            ast["where"].append(self.cond())
            while self.peek() == ("KW", "AND"):
                self.eat("KW", "AND"); ast["where"].append(self.cond())
        if self.peek() == ("KW", "GROUP"):
            self.eat("KW", "GROUP"); self.eat("KW", "BY")
            ast["group"] = self.eat("ID")
        if self.peek() == ("KW", "ORDER"):
            self.eat("KW", "ORDER"); self.eat("KW", "BY")
            col = self.eat("ID"); desc = False
            if self.peek()[0] == "KW" and self.peek()[1] in ("DESC", "ASC"):
                desc = (self.eat("KW") == "DESC")
            ast["order"] = (col, desc)
        if self.peek() == ("KW", "LIMIT"):
            self.eat("KW", "LIMIT"); ast["limit"] = int(self.eat("NUM"))
        if self.peek()[0] != "EOF":
            raise SyntaxError(f"多出來的東西：{self.peek()}")
        self.check_grouping(ast)
        return ast

    def select_list(self):
        if self.peek() == ("PUNC", "*"):
            self.eat("PUNC", "*"); return ["*"]
        items = [self.select_item()]
        while self.peek() == ("PUNC", ","):
            self.eat("PUNC", ","); items.append(self.select_item())
        return items

    def select_item(self):
        name = self.eat("ID")
        if self.peek() != ("PUNC", "("):
            return name
        func = name.upper()
        if func not in {"COUNT", "SUM", "AVG", "MIN", "MAX"}:
            raise SyntaxError(f"不支援的聚合函數：{name}")
        self.eat("PUNC", "(")
        if self.peek() == ("PUNC", "*"):
            self.eat("PUNC", "*"); arg = "*"
        else:
            arg = self.eat("ID")
        self.eat("PUNC", ")")
        if arg == "*" and func != "COUNT":
            raise SyntaxError("只有 COUNT 可以使用 *")
        if func == "COUNT" and arg != "*":
            raise SyntaxError("這個迷你子集只支援 COUNT(*)")
        return (func, arg)

    def check_grouping(self, ast):
        aggregates = [item for item in ast["select"] if isinstance(item, tuple)]
        plain_cols = [item for item in ast["select"] if isinstance(item, str)]
        if aggregates and ast["group"] is None:
            raise SyntaxError("這個迷你子集要求聚合查詢必須寫 GROUP BY")
        if ast["group"] is not None:
            if not aggregates:
                raise SyntaxError("GROUP BY 查詢至少要選一個聚合函數")
            if plain_cols != [ast["group"]]:
                raise SyntaxError("非聚合欄必須恰好是唯一的 GROUP BY 欄")
            if ast["order"] and ast["order"][0] != ast["group"]:
                raise SyntaxError("本版 GROUP BY 查詢只能依分組欄排序")

    def cond(self):
        col = self.eat("ID"); op = self.eat("OP")
        k, v = self.peek()
        if k not in ("NUM", "STR"):
            raise SyntaxError(f"比較的右邊要是常數，看到 {k}")
        self.i += 1
        return (col, op, v)

import json
ast = Parser(tokenize("SELECT name, year FROM student WHERE dept = '統計' AND year >= 3 ORDER BY year DESC LIMIT 2")).parse()
print(json.dumps(ast, ensure_ascii=False, indent=2))
print("\n→ 這棵「語法樹」就是 SQL 字串的結構化形式——引擎後續的一切都對著它做。")

In [ ]:
# 零件 3：volcano 執行器——每個運算子是一個 generator，一列一列往上「拉」
OPS = {"=": lambda a, b: a == b, "<>": lambda a, b: a != b, "!=": lambda a, b: a != b,
       "<": lambda a, b: a < b, ">": lambda a, b: a > b,
       "<=": lambda a, b: a <= b, ">=": lambda a, b: a >= b}

def scan_op(rows):
    for r in rows:
        yield r

def filter_op(src, conds):
    for r in src:
        if all(OPS[op](r[col], v) for col, op, v in conds):
            yield r

def sort_op(src, col, desc):                    # ⚠️ blocking operator：得先吸光下游才能吐第一列
    yield from sorted(src, key=lambda r: r[col], reverse=desc)

def limit_op(src, n):
    for i, r in enumerate(src):
        if i >= n:
            break
        yield r

def project_op(src, cols):
    for r in src:
        yield dict(r) if cols == ["*"] else {c: r[c] for c in cols}

def aggregate_label(item):
    func, arg = item
    return f"{func}({arg})"

def group_op(src, group_col, items):             # blocking：先收完每一組，才算得出聚合值
    groups = {}
    for row in src:
        groups.setdefault(row[group_col], []).append(row)
    for group_value, rows in groups.items():
        out = {}
        for item in items:
            if isinstance(item, str):
                out[item] = group_value
                continue
            func, arg = item
            values = rows if arg == "*" else [row[arg] for row in rows]
            if func == "COUNT":
                value = len(values)
            elif func == "SUM":
                value = sum(values)
            elif func == "AVG":
                value = sum(values) / len(values)
            elif func == "MIN":
                value = min(values)
            else:
                value = max(values)
            out[aggregate_label(item)] = value
        yield out

def run_sql(sql, db):
    ast = Parser(tokenize(sql)).parse()
    it = scan_op(db[ast["from"]])               # 由下而上組裝 pipeline
    if ast["where"]:
        it = filter_op(it, ast["where"])
    if ast["group"]:
        it = group_op(it, ast["group"], ast["select"])
    if ast["order"]:
        it = sort_op(it, *ast["order"])
    if ast["limit"] is not None:
        it = limit_op(it, ast["limit"])
    output_cols = ([aggregate_label(item) if isinstance(item, tuple) else item
                    for item in ast["select"]] if ast["group"] else ast["select"])
    return list(project_op(it, output_cols))

db = {"student": students, "takes": takes}
for r in run_sql("SELECT name, year FROM student WHERE dept = '統計' AND year >= 3", db):
    print(r)
print("\n🎉 你剛剛執行了自己引擎的第一句 SQL")

In [ ]:
# 加碼：我們自己的 EXPLAIN——把組好的管線印出來（跟 SQLite 的 EXPLAIN QUERY PLAN 同一個精神）
def explain_plan(sql):
    ast = Parser(tokenize(sql)).parse()
    steps = [f"scan({ast['from']})"]
    if ast["where"]:
        steps.append("filter(" + " AND ".join(f"{c}{op}{v!r}" for c, op, v in ast["where"]) + ")")
    if ast["group"]:
        aggs = ", ".join(aggregate_label(item) for item in ast["select"] if isinstance(item, tuple))
        steps.append(f"group({ast['group']}; {aggs})   ⚠️ blocking")
    if ast["order"]:
        steps.append(f"sort({ast['order'][0]}{' DESC' if ast['order'][1] else ''})   ⚠️ blocking")
    if ast["limit"] is not None:
        steps.append(f"limit({ast['limit']})")
    labels = [aggregate_label(item) if isinstance(item, tuple) else item for item in ast["select"]]
    steps.append("project(" + ", ".join(labels) + ")")
    print("執行計畫（由下而上）：")
    for i, op in enumerate(steps):
        print("  " * i + "└─ " + op)

explain_plan("SELECT name FROM student WHERE dept = '統計' AND year >= 3 ORDER BY year DESC LIMIT 2")
print()
print("同一句給 SQLite 解釋，對照著讀：")
for r in vcon.execute("""EXPLAIN QUERY PLAN SELECT name FROM student
                         WHERE dept = '統計' AND year >= 3 ORDER BY year DESC LIMIT 2"""):
    print("  ", r[3])
print("→ 一樣的骨架：掃描 → 過濾（藏在掃描裡）→ 排序（TEMP B-TREE ＝ 我們的 blocking sort）。")

In [ ]:
# 驗收：拿一批查詢跟 SQLite 對答案（自家引擎要跟業界標準一致才算數）
test_sqls = [
    "SELECT * FROM student WHERE dept = '統計'",
    "SELECT name FROM student WHERE year >= 3 AND dept <> '數學'",
    "SELECT sid, name FROM student ORDER BY sid DESC LIMIT 3",
    "SELECT sid, grade FROM takes WHERE grade > 85 ORDER BY grade DESC",
    "SELECT * FROM takes WHERE cid = 'C201' AND grade < 80",
]
for sql in test_sqls:
    mine = [tuple(r.values()) for r in run_sql(sql, db)]
    official = vcon.execute(sql).fetchall()
    same = (mine == official) if ("ORDER BY" in sql) else (sorted(map(str, mine)) == sorted(map(str, official)))
    print("✅" if same else "❌", sql)
    assert same
print("\n全部一致——我們的迷你引擎行為跟 SQLite 對齊（在這個小小的子集上）")

In [ ]:
# 引擎的錯誤訊息是哪來的？就是 parser 在喊——U02 那些 OperationalError 的源頭現形
for bad_sql in ["SELEC name FROM student",                 # 打錯關鍵字
                "SELECT name FROM student WHERE dept ==",  # 條件沒右邊
                "SELECT FROM student"]:                    # 忘了欄位
    try:
        Parser(tokenize(bad_sql)).parse()
    except SyntaxError as e:
        print(f"{bad_sql!r}\n   → SyntaxError: {e}")
print()
print("→ SQLite 的『near \"SELEC\": syntax error』就是它的 parser 版本的這句話。")
print("  現在你知道錯誤訊息裡的「near XXX」怎麼來的：parser 停在哪、就報哪。")

In [ ]:
# volcano 的精髓「懶惰拉取」現場：掃描器一邊喊、LIMIT 一邊拉——拉夠就停！
def noisy_scan(rows):
    for r in rows:
        print(f"　　（掃描器：撈出 {r['sid']}）")
        yield r

pipe = limit_op(filter_op(noisy_scan(students), [("year", ">=", 3)]), 2)
print("跟管線要第 1 列 →", next(pipe)["sid"])
print("跟管線要第 2 列 →", next(pipe)["sid"])
print("→ 六列的表只被掃了兩三列——LIMIT 的「快」不是魔法，是上游根本沒被拉。")
print("  但 ORDER BY 是 blocking：要先吸光才能吐第一列——「LIMIT 很快」在有排序時就失效，U07 的 TEMP B-TREE 呼應。")

## 1.4 mini engine v2：真正加入 `GROUP BY`（核心）

先把能力邊界寫成合約，再談成果。這一版刻意只做一個**清楚、可驗收的 SQL 子集**：

```text
SELECT group_col, aggregate [, aggregate ...]
FROM table [WHERE condition [AND condition ...]]
GROUP BY group_col [ORDER BY group_col [ASC|DESC]] [LIMIT n]

aggregate := COUNT(*) | SUM(col) | AVG(col) | MIN(col) | MAX(col)
```

| 這版真的會做 | 這版明確不做 |
|---|---|
| 單表、單一分組欄、先 `WHERE` 再分組 | 多欄分組、`HAVING`、`AS` 別名 |
| `COUNT(*)` 與四種數值聚合 | `NULL` 規則、運算式、aggregate 裡再套函數 |
| 以分組欄排序，再接 `LIMIT` | 以 aggregate 結果排序、join 後分組 |

限制不是偷偷漏做：parser 會主動拒絕超出合約的句子。前兩個零件已多認 `GROUP BY` 與函數呼叫；executor 則插入會先收齊輸入的 `group_op`。

In [ ]:
# 看 AST，也看真正跑出的分組結果：字串 → 結構 → blocking group_op → rows
group_sql = ("SELECT dept, COUNT(*), AVG(year) "
             "FROM student WHERE year >= 2 "
             "GROUP BY dept ORDER BY dept")
group_ast = Parser(tokenize(group_sql)).parse()
print(json.dumps(group_ast, ensure_ascii=False, indent=2))
print("\n執行計畫：")
explain_plan(group_sql)
print("\n答案：")
for row in run_sql(group_sql, db):
    print(row)

assert group_ast["group"] == "dept"
assert group_ast["select"][1] == ("COUNT", "*")
print("\n✅ parser 與 executor 都真的走到 GROUP BY")

In [ ]:
# 正向合約測試：不同聚合、WHERE、ORDER BY、LIMIT 都與 SQLite 對答案
group_test_sqls = [
    "SELECT dept, COUNT(*), MIN(year), MAX(year) FROM student GROUP BY dept ORDER BY dept",
    "SELECT cid, COUNT(*), SUM(grade), AVG(grade) FROM takes WHERE grade >= 75 GROUP BY cid ORDER BY cid",
    "SELECT year, COUNT(*), AVG(year) FROM student WHERE dept <> '資訊' GROUP BY year ORDER BY year LIMIT 2",
]
for sql in group_test_sqls:
    mine = [tuple(row.values()) for row in run_sql(sql, db)]
    official = vcon.execute(sql).fetchall()
    print("✅" if mine == official else "❌", sql)
    print("   mini =", mine)
    print("   SQLite =", official)
    assert mine == official
print("\n三條 GROUP BY 查詢全數對齊 SQLite ✅")

In [ ]:
# 負向合約測試：不支援的語法要明確失敗，不能悄悄算出錯答案
unsupported_sqls = [
    "SELECT dept, COUNT(*) FROM student",                         # 本版要求明寫 GROUP BY
    "SELECT dept FROM student GROUP BY dept",                    # 本版要求至少一個聚合
    "SELECT dept, name, COUNT(*) FROM student GROUP BY dept",    # name 既沒分組也沒聚合
    "SELECT dept, SUM(*) FROM student GROUP BY dept",            # 只有 COUNT 可吃星號
    "SELECT dept, COUNT(*) FROM student GROUP BY dept, year",    # 本版只做單欄分組
    "SELECT dept, COUNT(*) FROM student GROUP BY dept ORDER BY year",  # 只能依分組欄排序
    "SELECT dept, COUNT(*) FROM student GROUP BY dept HAVING COUNT(*) > 1",  # 尚無 HAVING
]
for sql in unsupported_sqls:
    try:
        Parser(tokenize(sql)).parse()
    except SyntaxError as exc:
        print(f"✅ 拒絕 {sql}\n   → {exc}")
    else:
        raise AssertionError(f"應拒絕卻接受：{sql}")
print("\n能力邊界也通過測試 ✅")

### 隨堂練習：blocking 還是 streaming？

`WHERE`／`LIMIT`／`ORDER BY`／`GROUP BY`／`DISTINCT`——哪些可以「來一列吐一列」？

<details><summary>答案</summary>
streaming：`WHERE`（逐列判斷）、`LIMIT`（數到 n 就喊停）。
blocking：`ORDER BY`（最後一列可能是最小的）、`GROUP BY`（最後一列可能改變任何一堆的聚合）、
`DISTINCT`（要記住看過誰——嚴格說是「半 blocking」：可以邊看邊吐，但要背著一個越長越大的 seen 集合）。
記憶體吃緊的就是 blocking 這幾位——大表 ORDER BY 沒索引時，引擎甚至要「外部排序」落地到磁碟。
</details>

### 【選讀】隨堂練習 A：繼續幫引擎加功能

任選一個（都是 5–10 行）：
1. 支援 `LIMIT n OFFSET m`（提示：`limit_op` 旁邊加一個 `offset_op` generator）；
2. 支援 `HAVING COUNT(*) >= n`（提示：在 `group_op` 後再接一個 filter）；
3. 讓 `WHERE` 支援 `OR`（提示：先拆 OR 再拆 AND——優先級！）。

<details><summary>參考解（選項 1）</summary>

```python
def offset_op(src, m):
    for i, r in enumerate(src):
        if i >= m:
            yield r
# parser 的 LIMIT 段加：
#   if self.peek() == ("KW", "OFFSET"): self.eat("KW","OFFSET"); ast["offset"] = int(self.eat("NUM"))
#  （記得把 OFFSET 加進 KEYWORDS 集合）
# run_sql(): if ast.get("offset"): it = offset_op(it, ast["offset"])   ← 要放在 limit_op 之前！
```
順序放錯（先 limit 再 offset）答案就錯——**運算子順序＝語意**，這正是最佳化器只敢做「等價」改寫的原因。
</details>

In [ ]:
# 練習 A 工作區：挑一個功能，改上面的零件（複製過來改也行），用 3 句測試驗收
# TODO





# 第 2 節：join 演算法、最佳化器與它會錯的方式

## 2.1 join 三演算法：相同任務、三種算法、天壤成本

| 演算法 | 想法 | 成本（A=n 列、B=m 列） |
|---|---|---|
| nested loop | A 每列 × 掃整個 B | O(n·m)——**平方級** |
| hash join | B 先蓋 hash 表，A 逐列查 | O(n + m) |
| sort-merge | 兩邊排好序、像拉鏈合併 | O(n log n + m log m) |

口說無憑，實測（3,000 × 3,000）：

In [ ]:
import random, time
random.seed(1)
A = [{"k": random.randrange(3000), "v": i} for i in range(3000)]
B = [{"k": random.randrange(3000), "w": i} for i in range(3000)]

def nested_loop(A, B):
    return [(a, b) for a in A for b in B if a["k"] == b["k"]]      # 900 萬次比較！

def hash_join(A, B):
    ht = {}
    for b in B:
        ht.setdefault(b["k"], []).append(b)
    return [(a, b) for a in A for b in ht.get(a["k"], [])]

def sort_merge(A, B):
    A2 = sorted(A, key=lambda r: r["k"]); B2 = sorted(B, key=lambda r: r["k"])
    out, j = [], 0
    for a in A2:
        while j < len(B2) and B2[j]["k"] < a["k"]:
            j += 1
        jj = j
        while jj < len(B2) and B2[jj]["k"] == a["k"]:
            out.append((a, B2[jj])); jj += 1
    return out

t = time.time(); r1 = nested_loop(A, B); t_nl = time.time() - t
t = time.time(); r2 = hash_join(A, B);   t_h  = time.time() - t
t = time.time(); r3 = sort_merge(A, B);  t_sm = time.time() - t
print(f"nested loop：{t_nl*1000:7.0f} ms")
print(f"hash join　：{t_h*1000:7.1f} ms（快 {t_nl/t_h:,.0f} 倍）")
print(f"sort-merge ：{t_sm*1000:7.1f} ms")
assert len(r1) == len(r2) == len(r3)
print(f"三種答案筆數一致（{len(r1):,} 列）✅ ——引擎挑演算法只影響快慢，不影響對錯")

In [ ]:
# sort-merge 的「拉鏈」到底怎麼拉？6×6 迷你版逐步直播（考卷上手演就是這樣）
A_mini = sorted([1, 3, 3, 5, 8, 9]); B_mini = sorted([2, 3, 5, 5, 9, 10])
print(f"A = {A_mini}\nB = {B_mini}\n")
i = j = 0
while i < len(A_mini) and j < len(B_mini):
    a, b = A_mini[i], B_mini[j]
    if a == b:
        print(f"A[{i}]={a} == B[{j}]={b} → 配對！雙方前進")
        i += 1; j += 1                      # 教學簡化：同值成串時真品會做「小回退」處理全部組合
    elif a < b:
        print(f"A[{i}]={a} <  B[{j}]={b} → A 前進")
        i += 1
    else:
        print(f"A[{i}]={a} >  B[{j}]={b} → B 前進")
        j += 1
print("\n→ 兩根手指各走一遍就結束：O(n+m)。排序的錢已經付過（或本來就排好）時，它是冠軍。")

In [ ]:
# 加映：nested loop 的「外圈選誰」學問——相同運算，小表當外圈 vs 大表當外圈
small = [{"k": i} for i in range(20)]
big = [{"k": random.randrange(20)} for _ in range(200_000)]
ht_big = {}
for r in big:
    ht_big.setdefault(r["k"], []).append(r)
ht_small = {}
for r in small:
    ht_small.setdefault(r["k"], []).append(r)

t = time.time()
n1 = sum(len(ht_big.get(a["k"], [])) for a in small)      # 外圈 20 次、內圈查 hash
t1 = time.time() - t
t = time.time()
n2 = sum(len(ht_small.get(b["k"], [])) for b in big)      # 外圈 20 萬次
t2 = time.time() - t
print(f"小表當外圈：{t1*1000:6.1f} ms；大表當外圈：{t2*1000:6.1f} ms（答案相同：{n1}={n2}）")
assert n1 == n2
print("→ 誰當外圈不影響答案、只影響成本——「join 順序」就是這件事的 n 張表版本（等下 2.3）。")

### 隨堂練習：三個場景各選哪個 join 演算法？

1. 兩張百萬列大表等值 join，記憶體夠。
2. 兩邊都已按 join 鍵排序（例如都是按日期存的流水）。
3. 20 列的維度表 join 百萬列交易表，交易表的 join 欄有索引。

<details><summary>答案</summary>
1. hash join（O(n+m)，蓋小表的 hash）。2. sort-merge——排序都省了，純拉鏈 O(n+m)。
3. NL＋內圈索引：小表當外圈 20 次、每次走樹 log m——這正是 SQLite 的日常（下一節）。
沒有全能冠軍：**資料的形狀（大小、順序、索引）決定誰上場**——引擎每次都重新選。
</details>

## 2.2 那 SQLite 用哪個？——nested loop，但**內圈走索引**

SQLite 的哲學：一律 nested loop（程式簡單、記憶體省），但把「內圈掃整表」換成「內圈走 B-tree」——
平方級瞬間變 O(n·log m)。所以 **U07 的 checklist 說 join 欄要有索引**，就是在救內圈。
（伺服器級的 PostgreSQL／DuckDB 會在 NL／hash／merge 之間動態挑。）

In [ ]:
#@title 📦 建 30 萬列訂單 ＋ 6 列城市維度表（本節的實驗場）
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
N = 300_000
orders_df = pd.DataFrame({
    "oid": np.arange(N),
    "city": rng.choice(["台中", "台北", "高雄", "台南", "新竹", "桃園"], N,
                       p=[.3, .25, .15, .12, .1, .08]),
    "area": None,   # 等下故意做「與 city 完全相關」的欄位（2.4 用）
    "amount": np.round(rng.lognormal(6, .8, N)).astype(int),
})
area_code = {"台中": "04", "台北": "02", "高雄": "07", "台南": "06", "新竹": "03", "桃園": "03A"}
orders_df["area"] = orders_df.city.map(area_code)

import os
if os.path.exists("w8.db"):
    os.remove("w8.db")
w8 = sqlite3.connect("w8.db")
orders_df.to_sql("orders", w8, index=False)
pd.DataFrame({"city": list(area_code), "region": ["中","北","南","南","北","北"]}).to_sql("dim_city", w8, index=False)
print(f"orders {N:,} 列、dim_city 6 列 就緒")

In [ ]:
def bench(con, sql, reps=5):
    t = time.time()
    for _ in range(reps):
        con.execute(sql).fetchall()
    return (time.time() - t) / reps * 1000

sql_join = "SELECT COUNT(*) FROM orders o JOIN dim_city d ON o.city = d.city WHERE d.region = '北'"
print("── join 欄沒索引 ──")
for r in w8.execute("EXPLAIN QUERY PLAN " + sql_join):
    print("  ", r[3])
t_slow = bench(w8, sql_join)

w8.execute("CREATE INDEX idx_o_city ON orders(city)")
w8.commit()
print("\n── join 欄建了索引 ──")
for r in w8.execute("EXPLAIN QUERY PLAN " + sql_join):
    print("  ", r[3])
t_fast = bench(w8, sql_join)
print(f"\n{t_slow:.0f} ms → {t_fast:.0f} ms")
print("→ 兩次都是 nested loop；差別是內圈從 SCAN 換成 SEARCH。也注意：引擎自動把「小表 dim_city」放外圈——")
print("  join 順序它自己挑（你寫 FROM orders JOIN dim_city 也一樣），依據就是 2.4 的統計。")

## 2.3 最佳化器的兩板斧：等價改寫

**① 謂詞下推（predicate pushdown）**：過濾越早做、中間結果越小。用 1.2 的代數四件套自己驗證：

In [ ]:
order_rows = orders_df.to_dict("records")[:60_000]          # 取 6 萬列做 Python 層實驗
city_rows = [{"city": c, "region": r} for c, r in
             zip(area_code, ["中", "北", "南", "南", "北", "北"])]

t = time.time()
slow_res = sel(join_h(order_rows, city_rows, "city"), lambda r: r["region"] == "北")   # 先 join 全部、再過濾
t_a = time.time() - t

t = time.time()
fast_res = join_h(order_rows, sel(city_rows, lambda r: r["region"] == "北"), "city")   # 先把維度表濾小、再 join
t_b = time.time() - t

print(f"先 join 再濾：中間結果 {len(order_rows):,} 列參與 join → {t_a*1000:.0f} ms")
print(f"先濾再 join：維度表剩 {len(sel(city_rows, lambda r: r['region']=='北'))} 列 → {t_b*1000:.0f} ms")
assert len(slow_res) == len(fast_res)
print("答案一致 ✅ ——「把 σ 往下推」是純代數改寫：結果不變、中間量大減。")
print("你在 U03 寫的「LEFT JOIN 條件放 ON」、CTE 先摺再接，其實都是在幫引擎做這件事。")

In [ ]:
# SQLite 也在做同一件事的證據：子查詢／view 外面的 WHERE，被「推」進去用索引
sql_sub = "SELECT * FROM (SELECT * FROM orders) WHERE city = '新竹'"       # 天真讀法：先掃全表再過濾
print("子查詢外的 WHERE →", "; ".join(r[3] for r in w8.execute("EXPLAIN QUERY PLAN " + sql_sub)))
w8.execute("DROP VIEW IF EXISTS v_north")
w8.execute("CREATE VIEW v_north AS SELECT o.*, d.region FROM orders o JOIN dim_city d ON o.city=d.city")
sql_v = "SELECT COUNT(*) FROM v_north WHERE city = '新竹'"
print("view 外的 WHERE　 →", "; ".join(r[3] for r in w8.execute("EXPLAIN QUERY PLAN " + sql_v)))
print()
print("→ 兩句都 SEARCH idx_o_city：外層的條件被攤平（flattening）＋下推進了子查詢/view。")
print("  所以「先包成 view 再過濾」通常不虧——引擎看穿了你的包裝。U03 用 view 定型報表的底氣在這。")

**② join 順序**：n 張表有 n! 種接法——

| 表數 | 順序數 |
|---|---|
| 3 | 6 |
| 6 | 720 |
| 10 | 3,628,800 |

順序不同、中間結果的大小可以差幾個數量級（先接會爆炸的一對＝災難）。
引擎不可能永遠把全部順序都跑一遍——它用**成本估計**挑一條看起來便宜的路。先把這句話變成可以輸入數字的試算器。

### 2.3.1 join 順序成本試算器（核心）

對目前中間結果 $L$ 接上一張表 $R$，先用最簡模型估：

$$|L \bowtie R| \approx |L| \times |R| \times \prod s_i$$

其中 $s_i$ 是新表與已接表之間各 join 條件的選擇率。下面的輸入只有兩份：

- `relation_cardinalities`：每張 relation 原始列數；
- `join_selectivities`：兩張 relation 配對後，笛卡兒積預計留下的比例。

這是**估計器，不是計時器**；`work` 只是各階段輸出列數總和，用來比較同一模型下的順序。它暫時假設均勻、獨立，也還沒算 hash 建表、索引 I/O 與記憶體成本——2.4 馬上拆這些假設為何會失準。

In [ ]:
# 可輸入的 left-deep join 順序成本試算器
from math import prod

def estimate_join_order(order, cardinalities, selectivities):
    if len(order) != len(set(order)) or set(order) != set(cardinalities):
        raise ValueError("order 必須把每張 relation 恰好列一次")
    if any(value < 0 for value in cardinalities.values()):
        raise ValueError("cardinality 不可為負")
    if any(not 0 <= value <= 1 for value in selectivities.values()):
        raise ValueError("selectivity 必須介於 0 與 1")

    joined = {order[0]}
    current_rows = float(cardinalities[order[0]])
    steps = []
    for relation in order[1:]:
        predicates = []
        for prior in joined:
            key = frozenset((prior, relation))
            if key in selectivities:
                predicates.append((prior, selectivities[key]))
        step_selectivity = prod(value for _, value in predicates) if predicates else 1.0
        input_rows = current_rows * cardinalities[relation]
        current_rows = input_rows * step_selectivity
        steps.append({
            "add": relation,
            "predicates": predicates,
            "input_pairs": input_rows,
            "output_rows": current_rows,
        })
        joined.add(relation)
    return steps

def show_join_order(order, cardinalities, selectivities):
    steps = estimate_join_order(order, cardinalities, selectivities)
    print(" → ".join(order))
    for step_no, step in enumerate(steps, 1):
        pred_text = (" × ".join(f"s({left},{step['add']})={value:g}"
                                for left, value in step["predicates"])
                     or "沒有 join 條件：CROSS，s=1")
        print(f"  step {step_no}: 接 {step['add']:<8} ｜ {pred_text:<42} "
              f"→ 中間結果約 {step['output_rows']:,.0f} 列")
    stats = {
        "final_rows": steps[-1]["output_rows"],
        "peak_rows": max(step["output_rows"] for step in steps),
        "work": sum(step["output_rows"] for step in steps),
    }
    print(f"  peak={stats['peak_rows']:,.0f}；work={stats['work']:,.0f}\n")
    return stats

# 100 位 vip 是 customer 的子集；sales 透過 customer 才連得到 vip
relation_cardinalities = {"sales": 10_000_000, "customer": 100_000, "vip": 100}
join_selectivities = {
    frozenset(("sales", "customer")): 1 / 100_000,
    frozenset(("customer", "vip")): 1 / 100_000,
}

large_first = show_join_order(
    ["sales", "customer", "vip"], relation_cardinalities, join_selectivities)
selective_first = show_join_order(
    ["vip", "customer", "sales"], relation_cardinalities, join_selectivities)

assert round(large_first["final_rows"]) == round(selective_first["final_rows"]) == 10_000
assert large_first["work"] > selective_first["work"] * 100
print(f"答案同為 10,000 列；先做高選擇性的 join，估計工作量少 "
      f"{large_first['work'] / selective_first['work']:,.0f} 倍 ✅")

In [ ]:
# 【選讀】只有三張表時可列舉全部順序；也會看見過早 CROSS 的爆炸
from itertools import permutations

ranked_plans = []
for candidate in permutations(relation_cardinalities):
    candidate_steps = estimate_join_order(
        list(candidate), relation_cardinalities, join_selectivities)
    ranked_plans.append({
        "order": " → ".join(candidate),
        "work": sum(step["output_rows"] for step in candidate_steps),
        "peak": max(step["output_rows"] for step in candidate_steps),
        "final": candidate_steps[-1]["output_rows"],
    })

for rank, plan in enumerate(sorted(ranked_plans, key=lambda item: item["work"]), 1):
    print(f"{rank}. {plan['order']:<30} "
          f"work={plan['work']:>15,.0f}  peak={plan['peak']:>15,.0f}  final={plan['final']:,.0f}")

assert len({round(plan["final"]) for plan in ranked_plans}) == 1
print("\n→ 六條路的最終列數相同；差別全發生在途中。表再多時 n! 暴增，才需要動態規劃或啟發式搜尋。")

## 2.4 成本估計靠統計——以及它會錯的兩種方式

U07 已看過 `ANALYZE` 寫進 `sqlite_stat1` 的數字（每個索引值平均對應幾列）。
問題來了：**「平均」是統計系最熟悉、也最會騙人的數字。**

### 錯法一：偏斜（skew）——同一個計畫，兩種命運

In [ ]:
# 造一個 99:1 的偏斜欄位
w8.execute("ALTER TABLE orders ADD COLUMN vip TEXT")
w8.execute("UPDATE orders SET vip = CASE WHEN oid % 100 = 0 THEN 'Y' ELSE 'N' END")
w8.execute("CREATE INDEX idx_vip ON orders(vip)")
w8.execute("ANALYZE"); w8.commit()

print("sqlite_stat1 對 idx_vip 的認知：",
      w8.execute("SELECT stat FROM sqlite_stat1 WHERE idx = 'idx_vip'").fetchone()[0],
      "←「平均每個值 15 萬列」")
for v in ("Y", "N"):
    n = w8.execute("SELECT COUNT(*) FROM orders WHERE vip = ?", (v,)).fetchone()[0]
    ms = bench(w8, f"SELECT SUM(amount) FROM orders WHERE vip = '{v}'", reps=3)
    print(f"vip = '{v}'：實際 {n:>7,} 列 → 同一個計畫跑 {ms:7.1f} ms")
print()
print("→ 平均 15 萬掩蓋了「Y 只有 3 千、N 有 29.7 萬」——引擎對兩者一視同仁，命運卻差百倍。")
print("  解方是直方圖（SQLite 的 STAT4 編譯選項、PostgreSQL 預設有）——沒有直方圖的世界，")
print("  偏斜查詢就要靠你自己心裡有數（統計系：你們比引擎懂偏斜！）。")

### 錯法二：相關性（correlation）——獨立性假設的滑鐵盧

多條件的選擇率，引擎用**獨立假設**連乘：P(A∧B) ≈ P(A)·P(B)。
但 `city` 與 `area`（區碼）**完全相關**——本例把邊際機率連乘，會低估約 3.3 倍：

In [ ]:
P_city = w8.execute("SELECT AVG(city = '台中') FROM orders").fetchone()[0]
P_area = w8.execute("SELECT AVG(area = '04') FROM orders").fetchone()[0]
actual = w8.execute("SELECT COUNT(*) FROM orders WHERE city = '台中' AND area = '04'").fetchone()[0]
indep_est = P_city * P_area * 300_000
print(f"P(city=台中) = {P_city:.3f}、P(area=04) = {P_area:.3f}")
print(f"獨立假設估計：{P_city:.3f} × {P_area:.3f} × 300,000 ≈ {indep_est:,.0f} 列")
print(f"實際　　　　：{actual:,} 列（兩欄根本是同一件事！）→ 低估 {actual/indep_est:.1f} 倍")
assert actual > indep_est * 2
print()
print("→ 估錯行數 → 挑錯 join 順序/演算法 → 慢查詢。這不是 bug，是「模型假設不成立」——")
print("  跟迴歸裡的共線性同一個味道。實務解方：別存衍生欄（U04 正規化！）、或提供多欄統計。")

## 2.5 DuckDB 列式引擎對決：換一種儲存，換一個宇宙

| | SQLite（列式 row store） | DuckDB（欄式 column store） |
|---|---|---|
| 一列的欄放一起？ | ✅（撈整列快—OLTP） | ❌ 同一欄放一起（掃整欄快—OLAP） |
| 執行方式 | 一列一列（volcano，今天造的） | **一批一批（向量化）**——迴圈開銷除以 2048 |
| 主場 | 點查、交易、App 後端 | 聚合、報表、資料分析 |

同一份 30 萬列，兩題各打一場：

In [ ]:
# 「向量化」是什麼感覺？一行 numpy 對一個 for 迴圈——DuckDB 的快，一半來自這個
arr10m = np.arange(10_000_000, dtype=np.int64)
t = time.time()
s1 = 0
for v in arr10m[:2_000_000]:                 # Python 迴圈（volcano 一列一列的縮影）：只跑 1/5 就好
    s1 += v
t_loop = (time.time() - t) * 5                # 換算成全量
t = time.time()
s2 = int(arr10m.sum())                        # 向量化：整批交給底層一次算
t_vec = time.time() - t
print(f"逐列迴圈（換算 1 千萬列）：{t_loop*1000:7.0f} ms")
print(f"整批向量化　　　　　　　：{t_vec*1000:7.1f} ms（快 {t_loop/t_vec:,.0f} 倍）")
print("→ 一列一列的開銷（函數呼叫、型別判斷）× 一千萬次 = 主要成本。")
print("  DuckDB 把 volcano 的「一次拉一列」改成「一次拉一批（2048 列）」——迴圈開銷除以 2048。")

In [ ]:
import sys, importlib.util, subprocess
if importlib.util.find_spec("duckdb") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
import duckdb

# 第一戰：分析題（整欄聚合）——DuckDB 的主場
agg = "SELECT city, COUNT(*), SUM(amount), AVG(amount) FROM orders GROUP BY city"
sq = bench(w8, agg, reps=5)
t = time.time()
for _ in range(5):
    duckdb.sql("SELECT city, COUNT(*), SUM(amount), AVG(amount) FROM orders_df GROUP BY city").fetchall()
dk = (time.time() - t) / 5 * 1000
print(f"整欄聚合 30 萬列：SQLite {sq:6.1f} ms ｜ DuckDB {dk:6.1f} ms → DuckDB 快 {sq/dk:.0f} 倍")

# 第二戰：點查（撈一列）——SQLite（帶索引）的主場
w8.execute("CREATE INDEX idx_oid ON orders(oid)"); w8.commit()
t_pt = bench(w8, "SELECT * FROM orders WHERE oid = 123456", reps=500)
t = time.time()
for _ in range(50):
    duckdb.sql("SELECT * FROM orders_df WHERE oid = 123456").fetchall()
t_pt_dk = (time.time() - t) / 50 * 1000
print(f"點查一列　　　　：SQLite {t_pt:6.3f} ms ｜ DuckDB {t_pt_dk:6.2f} ms → SQLite 快 {t_pt_dk/t_pt:.0f} 倍")

s_total = w8.execute("SELECT SUM(amount) FROM orders").fetchone()[0]
d_total = duckdb.sql("SELECT SUM(amount) FROM orders_df").fetchone()[0]
assert int(s_total) == int(d_total)
print(f"\n兩引擎總額一致（{s_total:,.0f}）✅ ——沒有誰比較對，只有誰的「儲存方向」對到你的工作負載。")
print("你的專題：App 用 SQLite（交易＋點查）、重報表可以像 U03 那樣丟給 DuckDB——兩個都帶走。")

In [ ]:
# DuckDB 也有 EXPLAIN——看一眼「另一個宇宙」的計畫長相（HASH_GROUP_BY、向量化掃描）
print(duckdb.sql("EXPLAIN SELECT city, SUM(amount) FROM orders_df GROUP BY city"))
print("→ 讀法跟 SQLite 同精神（由下而上）；出現 HASH_GROUP_BY／PROJECTION 等運算子，")
print("  每個節點一次處理一批（vector）。換引擎，計畫的「方言」不同、骨架相同——你已經會讀了。")

### 2.5.1【選讀】DuckDB `EXPLAIN ANALYZE`：把實際列數與時間貼回計畫

`EXPLAIN` 只展示預計採用的 physical plan；`EXPLAIN ANALYZE` 會**真的執行查詢**，再把各 operator 的實際輸出列數與耗時標回樹上。閱讀時由下往上找：scan 讀了多少 → filter 留下多少 → group 壓成幾組 → order 收到幾列。

> DuckDB 版本、硬體、快取與 pandas 掃描介面都可能改變 operator 名稱、排版和秒數。所以下格只觀察結構與數量級，**刻意不對 plan 文字或時間寫精確 assert**；若教室畫面和講義長得不完全相同，是正常的。

In [ ]:
# EXPLAIN ANALYZE 會執行查詢；輸出排版與計時是版本敏感資訊
profile_sql = ("SELECT city, COUNT(*) AS order_count, SUM(amount) AS revenue "
               "FROM orders_df WHERE amount >= 1000 "
               "GROUP BY city ORDER BY city")
profile_rows = duckdb.sql("EXPLAIN ANALYZE " + profile_sql).fetchall()

print("DuckDB version:", duckdb.__version__)
for profile_row in profile_rows:
    print("\n".join(str(part) for part in profile_row))

profile_result = duckdb.sql(profile_sql).fetchall()
print(f"\n查詢實際回傳 {len(profile_result)} 組；前 3 組：", profile_result[:3])
print("讀圖任務：圈出 scan → filter → group → order，記下各層列數如何縮小。")
print("本格不比對 operator 名稱、完整 plan 文字或精確秒數。")

In [ ]:
# 「LIMIT 很快」的邊界實測：streaming 的 LIMIT 秒回，blocking（排序）的 LIMIT 照樣全表買單
t_stream = bench(w8, "SELECT * FROM orders WHERE amount > 100 LIMIT 5", reps=50)       # 掃到 5 列就停
t_block  = bench(w8, "SELECT * FROM orders ORDER BY amount DESC LIMIT 5", reps=5)      # 先吸光 30 萬列排序
print(f"WHERE … LIMIT 5      ：{t_stream:8.3f} ms（上游拉幾列就停）")
print(f"ORDER BY … LIMIT 5   ：{t_block:8.1f} ms（排序是 blocking：LIMIT 幫不上）")
print("計畫對照：", "; ".join(r[3] for r in w8.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM orders ORDER BY amount DESC LIMIT 5")))
assert t_block > t_stream * 5
print("→ 救法你在 U07 學過：給 amount 建索引，讓「排序」變成「沿樹走」——blocking 變 streaming。")

### 隨堂練習 B（3 分鐘，紙上）：畫出代數樹

把這句翻成 σ π ⋈ γ 的樹（由下而上）：
`SELECT c.city, SUM(o.amount) FROM orders o JOIN customers c ON o.cid=c.cid WHERE c.age >= 30 GROUP BY c.city`

<details><summary>參考答案</summary>

```
 γ_{city, SUM(amount)}
        │
        ⋈_{o.cid = c.cid}
       ╱  ╲
 orders    σ_{age ≥ 30}
              │
          customers
```
σ 推到 customers 上面（join 之前）——謂詞下推後 join 的右邊變小。若把 σ 放在 ⋈ 之後也「對」，只是貴。
</details>

### 最佳化器不是神：你還能做的三件事

1. **餵好統計**：大量增刪後 `ANALYZE`；偏斜與相關欄位心裡有數（它沒有直方圖，你有）。
2. **寫好等價形**：U07 的「函數別包欄位」、U02 的「NOT IN 換 NOT EXISTS」——別逼它做它不會的改寫。
3. **蓋好路**：join 欄與高頻條件的索引——它只能在「現有的路」裡挑最快的一條。

## 2.6【AI 協作】把 EXPLAIN 丟給 AI 之前

AI 很會解釋計畫，但你要**餵對料、問對事**：

> 這是 SQLite 的 EXPLAIN QUERY PLAN 輸出：（貼輸出）
> schema 與索引如下：（貼 `SELECT sql FROM sqlite_master`）
> 資料量：orders 30 萬列、dim_city 6 列。
> 請回答：1) 外圈是誰、為什麼？ 2) 哪一步最貴？ 3) 若要加一個索引，加哪個、預期計畫怎麼變？

**驗收**：它說要加的索引，你**建了再跑一次 EXPLAIN**——計畫沒變成它說的樣子，就把新輸出貼回去追問。
（沒有 schema 與資料量，AI 只能瞎猜——跟最佳化器沒有統計就亂估，是同一個道理。）

# 課堂實作（35 分鐘）

## A. 計畫閱讀三題（10 分）：先預測外圈/內圈與用不用索引，再跑 EXPLAIN

In [ ]:
plan_sqls = [
    "SELECT * FROM orders o JOIN dim_city d ON o.city = d.city WHERE o.amount > 5000",
    "SELECT d.region, SUM(o.amount) FROM orders o JOIN dim_city d ON o.city = d.city GROUP BY d.region",
    "SELECT * FROM orders WHERE vip = 'Y' AND city = '台中'",
]
for i, sql in enumerate(plan_sqls, 1):
    print(f"Q{i}: {sql}")
    # 預測完取消下一行註解：
    # print("   →", "; ".join(r[3] for r in w8.execute("EXPLAIN QUERY PLAN " + sql)))
print("\n提示：誰當外圈？內圈有沒有索引可走？（解答就在你取消註解的那一刻）")

## B. join 量測（10 分）：把 2.1 的 A、B 改成 10,000 × 10,000

先估：nested loop 會變幾倍慢？（提示：平方級）hash 呢？跑下去對答案；順手把三個函數的結果再 assert 一次。

## C. 把 EXPLAIN 用回你的專題（15 分）

1. 挑專題裡「跨表最多」的報表查詢 → `EXPLAIN QUERY PLAN`；
2. 回答三個問題寫進 notebook：外圈是哪張表？每個內圈有沒有 SEARCH？有沒有 TEMP B-TREE？
3. 若有 SCAN 大表或加班排序——用 U07 的 checklist 開藥；改完前後計時對照。

In [ ]:
# 實作 B 工作區：10,000 × 10,000 的三演算法對決（先寫下預測再跑）
random.seed(2)
A_big = [{"k": random.randrange(10_000), "v": i} for i in range(10_000)]
B_big = [{"k": random.randrange(10_000), "w": i} for i in range(10_000)]
# 預測：NL 會是 3,000×3,000 的 ___ 倍慢；hash 大約 ___ 倍
# TODO：計時三個函數（nested_loop 那行跑之前先想想要等多久…）





In [ ]:
# 課堂實作工作區
mycon = sqlite3.connect("myapp.db")

# TODO：你的報表查詢 → EXPLAIN QUERY PLAN → 三個問題 → （需要的話）建索引 → 前後計時
print("工作區就緒")

## 專題進度建議（非繳交）

**U08–U09 是組裝期**——三分頁組好、5 張報表齊、8 個 assert 綠、demo 排練起來。

1. 報表全部過一次 EXPLAIN（今天實作 C），慢的修掉；
2. 對照共同要求 10 條逐項打勾，缺的排進本週；
3. 開始寫 demo 腳本（點什麼、說什麼、亮點在哪）——下個單元報告規範課會用到；
4. 錄備援：把 demo 流程截圖或錄 30 秒（下個單元教你怎麼排）。

# 本單元你應該帶走

1. SQL 的一生：字串 →（tokenizer）詞 →（parser）AST →（代數＋最佳化）計畫 →（volcano）一列一列拉；mini engine 已真的支援受限的單欄 `GROUP BY`，且正反向合約都有測。
2. 關聯代數 σπ⋈γ 吃表吐表、可任意套疊——最佳化器做的是「等價改寫」：謂詞下推（連 view 都會被攤平下推）、挑 join 順序。
3. join 三演算法：NL 平方級、hash O(n+m)、merge 排序後拉鏈；SQLite＝NL＋內圈索引——join 欄建索引是救命的；資料形狀決定誰上場。
4. blocking vs streaming：ORDER BY／GROUP BY 要吸光才能吐——「LIMIT 很快」遇到排序就失效。
5. 成本估計靠統計，而統計會錯的兩種方式你最熟：**偏斜**（平均騙人）與**相關**（獨立假設崩壞）。
6. 列式＋向量化讓分析快 10–100 倍；點查與交易仍是列存＋索引的天下——**工作負載決定引擎**；DuckDB 的 `EXPLAIN ANALYZE` 還會把實際列數與時間標回計畫。

**下個單元**：交易與復原——ACID、兩條連線重現三種並行異常、鎖與 MVCC、**當機模擬與 WAL replay**；外加現代資料庫速覽與★報告規範。讀物：Silberschatz ch15–16（本單元）、ch17–19 選讀（預習）；Ullman ch15–16。

---
## 附錄 A：本單元 cheatsheet

```
SQL 流水線：tokenize → parse(AST) → 代數 → 最佳化（下推、join 順序、挑索引）→ volcano 執行
代數四件套：σ 挑列（WHERE）  π 挑欄（SELECT）  ⋈ 接表（JOIN）  γ 分堆（GROUP BY）
mini GROUP BY：單表＋單分組欄；COUNT(*)/SUM/AVG/MIN/MAX；先 WHERE，再 group（blocking）
join 成本：NL O(nm)｜NL+內圈索引 O(n log m)｜hash O(n+m)｜merge O(n log n + m log m)
順序試算：新中間量 ≈ 舊中間量 × 新表列數 × 相關選擇率；比較 peak 與 work
SQLite join：一律 NL；小表當外圈、內圈盡量 SEARCH（join 欄要有索引！）
計畫閱讀三問：外圈誰？內圈 SEARCH 嗎？有沒有 TEMP B-TREE？
估計的兩個天敵：偏斜（平均≠個案；解=直方圖/心裡有數）
              相關（P(A∧B)≠P(A)P(B)；解=別存衍生欄）
分析快法：欄式儲存＋向量化（DuckDB）；點查交易：列存＋B-tree（SQLite）
```

### 附錄 A2：mini 引擎延伸地圖（想玩下去的人）

| 想加的功能 | 動哪個零件 | 提示 |
|---|---|---|
| 不分組的全表 `COUNT(*)` | 放寬 grouping 合約、建立單一隱形群組 | 空表也要回傳 0 |
| 多欄 `GROUP BY`／`HAVING` | group key 改 tuple、聚合後再 filter | 注意 SQL 執行順序 |
| `JOIN … ON` | parser 支援兩表、executor 接 join_h() | 先做等值 join 就好 |
| `OR` 與括號 | cond() 改成「or 層 → and 層 → 原子」三層遞迴 | 優先級＝文法層級 |
| 最佳化器 v0 | run_sql() 前把 σ 往 scan 推 | 你已經懂謂詞下推 |

全套加完 ≈ 300 行——這就是 `extra_modern.ipynb` 之後的自學路線；資訊系一學期的編譯器＋資料庫課，你已經摸到骨架。

## 附錄 B：讀物地圖（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| 查詢處理總覽、代數 | §15.1–15.3、§2.6 | §15.1–15.2、§16.1 |
| join 演算法 | §15.5–15.6 | §15.4–15.6 |
| 最佳化、等價改寫 | §16.1–16.3 | §16.2–16.3 |
| 成本估計與統計 | §16.4 | §16.4 |
| 列式／向量化 | §13.6、§24.4 | —— |